# Named Entity Recognition and Classification
## Install/import libraries

In [ ]:
from collections import Counter 
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import svm
import gensim
from nltk.corpus.reader import ConllCorpusReader

## Data exploration

In [ ]:
# Load the IPM NEL dataset
path_to_train_folder = "training_sets/sentiment-topic/"
conll_path = f"{path_to_train_folder}ipm_nel_corpus/"
conll_train = ConllCorpusReader(conll_path, 'ipm_nel.conll', ['words', 'ignore', 'ne', 'pos'])

## Create features

We create the features for the training and test datasets.

In [ ]:
training_features = []
training_gold_labels = []

"""
for token, pos, ne_label in conll_train.iob_words():
    training_feature = {
       'token': token,
       'pos': pos
    }

    training_features.append(training_feature)
    training_gold_labels.append(ne_label)
"""

In [ ]:
test_features = []
test_gold_labels = []

"""
for token, pos, ne_label in ...:
    test_feature = {
       'token': token,
       'pos': pos
    }

    test_features.append(test_feature)
    test_gold_labels.append(ne_label)
"""

## Descriptive statistics
Frequency distribution of the NERC labels in the training and test data:

In [ ]:
train_label_counts = Counter(training_gold_labels)
test_label_counts = Counter(test_gold_labels)

print("NERC label frequency in training data:")
for label, count in train_label_counts.most_common():
    print(f"{label}: {count}")

print("\nNERC label frequency in test data:")
for label, count in test_label_counts.most_common():
    print(f"{label}: {count}")

Analyzing training and test data (im)balance:

In [ ]:
train_total = sum(train_label_counts.values())
test_total = sum(test_label_counts.values())

print("\nPercentage distribution in training data:")
for label, count in train_label_counts.most_common():
    percentage = (count / train_total) * 100
    print(f"{label}: {percentage}%")

print("\nPercentage distribution in test data:")
for label, count in test_label_counts.most_common():
    percentage = (count / test_total) * 100
    print(f"{label}: {percentage}%")

#TODO: explain distribution

## Vectorize data
For more balanced performance across entity types and shorter model training time, we use Google's pre-trained Word2Vec word embedding model to obtain vectors for our training and test features.

In [ ]:
# Load word embedding model
gensim_path = 'GoogleNews-vectors-negative300.bin\\GoogleNews-vectors-negative300.bin'
word_embedding_model = gensim.models.KeyedVectors.load_word2vec_format(gensim_path, binary=True)

# Initialize arrays
training_vec = []
test_vec = []

# Obtain vectors for training features
for token_dict in training_features:
    word = token_dict['token']

    # Is word in the model vocabulary (loaded with the Google word2vec embeddings)?
    if word in word_embedding_model:
        vector = word_embedding_model[word]  # assign embedding vector value to variable
    else: 
        vector = [0] * 300  # Create a vector with 300 zeros, as word2vec model has 300 dimensions

    training_vec.append(vector)

# Obtain vectors for test features
for token_dict in test_features:
    word = token_dict['token']

    # Is word in the model vocabulary (loaded with the Google word2vec embeddings)?
    if word in word_embedding_model:
        vector = word_embedding_model[word]  # assign embedding vector value to variable
    else: 
        vector = [0] * 300  # Create a vector with 300 zeros as word2vec model has 300 dimensions

    test_vec.append(vector)

## Training the Support Vector Machine
Since the training data contains a varying amount of instances across domains, there is a need for a model that is robust against overfitting. SVMs provide robustness via the use of a regularization parameter, often denoted as `C`. Furthermore, SVMs are based on the maximization of margins between classes. This often leads to good generalization performance, which is crucial for our task of minimizing misclassifications (especially across similar entity types). 

We choose the linear kernel, as the linear kernel of SVMs may outperform the polynomial and RBF kernel for the NER task (Alokaili and Menai, 2019).

In [ ]:
lin_clf = svm.LinearSVC(kernel='linear')

# NOTE: DON'T run this cell again if you've already trained the SVM! It may take up to 10 minutes!
lin_clf.fit(training_vec, training_gold_labels)

## Make predictions for the test set and evaluate
Make predictions with the model (predict the NER labels of the tokens in the test set).

In [ ]:
# Obtain predictions from the SVM model
svm_predictions = lin_clf.predict(test_vec)

# Generate the classification report and print it
class_report = classification_report(test_gold_labels, svm_predictions)
print(class_report)

#TODO: Analyze classification report

* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?
